What is LSTM? 

LSTM: Long Short Term Memory is a neural network architecture that allows RNNs to retain information over long sequences. It has a gradient highway where the previous layer's output is passed through and added to the processed input token. The cell state allows information to persist. There are 3 gating mechanisms thea control the flow of information. A forget gate which decides whther to discard the information from the previous cell state. Input gate that determines which new information to store in the cell state. Output gate that decides what part of the cell state to output as the hidden state/, 


Gates that matter for volatility forecasting 
Forget gate is important as it allows us to just focus on the volatility numbers that matter such as sudden drops or sudden increases in prices. 
input gate is also important as it determines whether the current input is important

We need to feed into LSTM log_returns as we need continuous data about the exact price change of a feature. We also need to add volatility as it captures the recent turbulence and variability the market is in right now. Using both it can give us a context of how th emarket is right now. Raw prices would not work as othe values arent normalized.

The tradeoffs of using a longer window would be that we have more data about how the feature moves along time. Thus, we can determine if an abrupt change in price or high volatility is common or is uncommon. Using shorter windows allows us to determine the best case scenario at the moment like a greedy algorithm. I think the LSTm for bitcoin should lookback around 30 days. 

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
import sys
import os

In [3]:
def create_sequences(df: pd.DataFrame, window_size: int):
    X_data = df[['log_returns','volatility']].values
    y_data = df['realized_variance'].values
    X = [X_data[i:i+window_size] for i in range(len(df)-window_size)]
    y = [y_data[i+window_size] for i in range(len(df)-window_size)]
    
    return torch.tensor(np.array(X), dtype = torch.float32), torch.tensor(np.array(y), dtype = torch.float32)

In [4]:
# load csv 
btc_df = pd.read_csv('../data/btc_data.csv', index_col = 'Date', parse_dates = True)

train = btc_df.loc["2018-01-31":"2022-12-31"]
val = btc_df.loc["2023-01-01":"2023-12-31"]
test = btc_df.loc["2024-01-01":"2024-12-31"]

print(btc_df.index)

DatetimeIndex(['2018-01-31', '2018-02-01', '2018-02-02', '2018-02-03',
               '2018-02-04', '2018-02-05', '2018-02-06', '2018-02-07',
               '2018-02-08', '2018-02-09',
               ...
               '2024-12-21', '2024-12-22', '2024-12-23', '2024-12-24',
               '2024-12-25', '2024-12-26', '2024-12-27', '2024-12-28',
               '2024-12-29', '2024-12-30'],
              dtype='datetime64[us]', name='Date', length=2526, freq=None)


In [5]:
def scale_data(train_df, val_df, test_df):
    log_scaler = StandardScaler()
    vol_scaler = StandardScaler()
    target_scaler = StandardScaler()
    
    train_log_return = log_scaler.fit_transform(train_df[['log_returns']])
    train_vol = vol_scaler.fit_transform(train_df[['volatility']])
    train_target = target_scaler.fit_transform(train_df[['realized_variance']])
    
    def build_df(df, log_data, vol_data, target_data):
        return pd.concat([
            pd.DataFrame(log_data, index=df.index, columns=['log_returns']),
            pd.DataFrame(vol_data, index=df.index, columns=['volatility']),
            pd.Series(target_data.flatten(), index=df.index, name='realized_variance')
        ], axis=1)
    
    scaled_train = build_df(train_df, train_log_return, train_vol, train_target)
    scaled_val = build_df(val_df, log_scaler.transform(val_df[['log_returns']]), vol_scaler.transform(val_df[['volatility']]), target_scaler.transform(val_df[['realized_variance']]))
    scaled_test = build_df(test_df, log_scaler.transform(test_df[['log_returns']]), vol_scaler.transform(test_df[['volatility']]), target_scaler.transform(test_df[['realized_variance']]))
    
    return scaled_train, scaled_val, scaled_test, target_scaler

In [6]:
# Scale data 
scaled_train, scaled_val, scaled_test, target_scaler = scale_data(train, val, test)

In [7]:
#Creating train, val, and test sets 

X_train, y_train = create_sequences(scaled_train, window_size = 30)
X_test, y_test = create_sequences(scaled_test, window_size = 30)
X_val, y_val = create_sequences(scaled_val, window_size = 30)

print(f"X_train shape: {X_train.shape}\ny_train shape: {y_train.shape}")


X_train shape: torch.Size([1766, 30, 2])
y_train shape: torch.Size([1766])


In [8]:
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

if project_root not in sys.path:
    sys.path.append(project_root)
    
from models.lstm import LSTMModel


input_size = 2 # Two inputs volatility and log returns
hidden_size = 64 # Number of units in the hidden state
num_layers = 2 # Two stacked LSTMs
dropout = 0.2 # Dropout rate of 20%
learning_rate = 1e-3

#Creating LSTM model
model = LSTMModel(input_size = input_size, 
                       hidden_size = hidden_size,
                       num_layers = num_layers, 
                       dropout = dropout)

# MSE loss function 
loss_fn = nn.MSELoss()
# Adam Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [9]:
# Setting up dataloaders
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, shuffle = True, batch_size = 128)
test_loader = DataLoader(test_ds, batch_size = 128)
val_loader = DataLoader(val_ds, batch_size = 128)


In [14]:
def train_and_validate(model, train_loader, val_loader, optimizer, loss_fn, save_path, num_epochs = 50):
    
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path))
        model.eval()
        print("Loaded from checkpoint")
    else:
        #Training loop 
        print("Training from scratch")
        model.train()

        train_losses = []
        val_losses = []

        for epoch in range(num_epochs):
            #--- Train ---
            model.train()
            running_train_loss = 0
            for X_batch, y_batch in train_loader:
                
                X_batch, y_batch = X_batch.float(), y_batch.float().view(-1,1)
                #Forward pass
                pred = model(X_batch)
                
                
                loss = loss_fn(pred, y_batch)
                
                #Backprop and optimization 
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                running_train_loss += loss.item()
            
            avg_train_loss = running_train_loss/len(train_loader)
            train_losses.append(avg_train_loss)
                
            #Validation
            model.eval()
            running_val_loss = 0
            with torch.no_grad(): # Used as no gradient computation is needed since there is no backprop on validation
                for X_b_val, y_b_val in val_loader:
                    
                    X_b_val, y_b_val = X_b_val.float(), y_b_val.float().view(-1,1)
                    
                    #Forward pass
                    val_pred = model(X_b_val)
                    
                    v_loss = loss_fn(val_pred, y_b_val)
                    
                    running_val_loss += v_loss.item()
                
                avg_val_loss = running_val_loss/len(val_loader)
                val_losses.append(avg_val_loss)
            
            if (epoch + 1) % 10 == 0 or epoch == 0 or (epoch+1) % 50 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] | "
                    f"Train Loss: {avg_train_loss:.6f} | "
                    f"Val Loss: {avg_val_loss:.6f}")
        #Save model
        torch.save(model.state_dict(), save_path)
        return model
        

In [11]:
def qlike_loss(actual, pred, epsilon = 1e-10):
    actual = np.clip(actual, epsilon, None)
    pred = np.clip(pred, epsilon, None)
    
    loss = (actual/pred) - np.log(actual/pred) -1
    
    return loss.mean()

In [ ]:
def evaluate_model(model, test_loader, test_df, target_scaler, window_size=30):
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for X_btest, _ in test_loader:
            val_pred = model(X_btest.float())
            predictions.append(val_pred)
            
    lstm_forecast_scaled = torch.cat(predictions).cpu().numpy()
    actual_scaled = test_df['realized_variance'].iloc[window_size:].values.reshape(-1,1)
    
    lstm_forecast_real = target_scaler.inverse_transform(lstm_forecast_scaled).flatten()
    actual_real = target_scaler.inverse_transform(actual_scaled).flatten()
    
    forecast_index = test_df.index[window_size:]
    results = pd.DataFrame({'actual': actual_real, 'lstm_forecast': lstm_forecast_real}, index=forecast_index)
    
    mse = mean_squared_error(results['actual'], results['lstm_forecast'])
    mae = mean_absolute_error(results['actual'], results['lstm_forecast'])
    qlike = qlike_loss(results['actual'].values, results['lstm_forecast'].values)

    
    print(f"MSE: {mse:.6f} | MAE: {mae:.6f}| QLIKE: {qlike:.6f}")
    return results

In [15]:
# MSE LOSS LSTM 
save_path = '../models/mse_lstm_model.pth'
model = LSTMModel(input_size=2, hidden_size=64, num_layers=2, dropout=0.2)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
loss_fn = nn.MSELoss()
model = train_and_validate(model = model, train_loader= train_loader, val_loader = val_loader, optimizer = optimizer, loss_fn = loss_fn, save_path=save_path)

results = evaluate_model(model, test_loader, scaled_test, target_scaler)


Training from scratch
Epoch [1/50] | Train Loss: 1.437861 | Val Loss: 0.508004
Epoch [10/50] | Train Loss: 0.981778 | Val Loss: 0.070465
Epoch [20/50] | Train Loss: 0.983027 | Val Loss: 0.069471
Epoch [30/50] | Train Loss: 0.982460 | Val Loss: 0.068933
Epoch [40/50] | Train Loss: 0.985057 | Val Loss: 0.068641
Epoch [50/50] | Train Loss: 0.983564 | Val Loss: 0.068457
MSE: 0.000003 | MAE: 0.001308| QLIKE: 1.806592
